# 02 — External validation, robustness, and final interpretation

This notebook is deliberately gated until the H1-N comparison is frozen. It prepares the licensed external protocol, evaluates robustness without tuning, aggregates completed results, and records limitations. It must not be used to select an architecture, threshold, or seed.

**Scope:** confirmatory transfer and robustness only after the documented freeze.

## 1. Locked external validation and robustness


This notebook is intentionally a **PENDING / read-only** gate until the full H1-N comparison is complete and its frozen evaluation plan is recorded. It does not use external data for training, early stopping, threshold selection, representation selection, or robustness tuning.

In [ ]:
from pathlib import Path
import json

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL


def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'ai_image_detector').is_dir():
            return candidate
    raise RuntimeError('Open Jupyter from this repository or one of its subdirectories.')


REPO = find_repository_root()
ARTIFACT_ROOT = REPO / 'artifacts'
print(f'Repository: {REPO}')

def completed_h1n_runs() -> list[dict[str, object]]:
    rows = []
    for run_path in ARTIFACT_ROOT.glob('*/run.json'):
        metrics_path = run_path.parent / 'internal_test_metrics.json'
        if not metrics_path.is_file():
            continue
        run = json.loads(run_path.read_text(encoding='utf-8'))
        if run.get('preprocessing', {}).get('protocol') != CONTROLLED_PREPROCESSING_PROTOCOL:
            continue
        config = run.get('config', {})
        rows.append({
            'run': run_path.parent.name,
            'representation': config.get('representation'),
            'seed': config.get('seed'),
            'threshold': run.get('threshold'),
            'metrics_path': str(metrics_path),
        })
    return sorted(rows, key=lambda row: (str(row['representation']), int(row['seed'])))

internal_runs = completed_h1n_runs()
if internal_runs:
    display(pd.DataFrame(internal_runs))
else:
    print('PENDING: no completed controlled H1-N internal result is available.')

## External locked-test plan

1. Complete the six predeclared H1-N internal runs and their cluster-aware analysis. Do not choose a winning seed.
2. Freeze the representation comparison, three-seed aggregation, checkpoint selection rule, validation-selected threshold rule, and common-raster preprocessing.
3. Obtain Synthbuster and RAISE-1k under their research licences, record the local archive hashes, and create a separate external-only manifest. The preparation command never downloads data.
4. Audit exact-file and perceptual-hash candidate overlap against Defactify before inference. Any overlap handling is documented before metrics are calculated.
5. Run the frozen model exactly once on the external manifest, store every prediction, and report generator-specific, macro and worst-generator values without changing the model or validation-selected threshold. Stratify each synthetic generator by the prepared manifest's explicit `defactify_train_relation`: same-named generator, same-family/different or unspecified version, or unseen family. Do not infer these relations from names or describe all Synthbuster generators as unseen.

Because the Defactify test rows were already read during D0, H1-N metrics there are exploratory internal stress tests. The external corpus is the confirmatory evaluation; a good internal score cannot replace it.

In [ ]:
import shlex
import sys

def external_manifest_command(
    synthetic_root: Path,
    raise_root: Path,
    output_root: Path,
) -> list[str]:
    """Build the manifest only after the H1-N plan is frozen and local data are licensed."""
    return [
        sys.executable,
        'scripts/prepare_synthbuster_external.py',
        '--synthetic-root', str(synthetic_root),
        '--raise-root', str(raise_root),
        '--output-root', str(output_root),
        '--reference-manifest', str((REPO / 'data/processed/defactify_grouped/manifest.csv').relative_to(REPO)),
    ]

print('External manifest preparation is locked: configure real, licensed local paths only after freezing H1-N.')
print('The function above returns a valid command once its three Path arguments are supplied.')

## Frozen external evaluation command

This command is unavailable until both a documented frozen experiment and a locally prepared, licensed external manifest exist. It evaluates once; it must not be used for model or threshold selection.

In [ ]:
import os
import shlex
import subprocess
import sys


def external_evaluation_command(frozen_experiment: str, external_manifest: Path) -> list[str]:
    if Path(frozen_experiment).name != frozen_experiment:
        raise ValueError('Use one experiment directory name, not a path.')
    manifest_path = external_manifest.resolve()
    try:
        manifest_argument = str(manifest_path.relative_to(REPO))
    except ValueError:
        manifest_argument = str(manifest_path)
    experiment_dir = REPO / 'artifacts' / frozen_experiment
    return [
        sys.executable,
        'scripts/evaluate_external.py',
        '--manifest', manifest_argument,
        '--experiment-dir', str(experiment_dir.relative_to(REPO)),
        '--output-dir', str((experiment_dir / 'external').relative_to(REPO)),
        '--bootstrap-repeats', '2000',
        '--bootstrap-seed', '20260829',
    ]


FROZEN_EXPERIMENT = os.environ.get('H1N_FROZEN_EXPERIMENT')
external_manifest_value = os.environ.get('H1N_EXTERNAL_MANIFEST')
if FROZEN_EXPERIMENT and external_manifest_value:
    external_manifest = Path(external_manifest_value)
    if not external_manifest.is_absolute():
        external_manifest = REPO / external_manifest
    EXTERNAL_EVALUATION_COMMAND = external_evaluation_command(FROZEN_EXPERIMENT, external_manifest)
    print(shlex.join(EXTERNAL_EVALUATION_COMMAND))
    if os.environ.get('RUN_H1N_EXTERNAL_EVALUATION') == '1':
        subprocess.run(EXTERNAL_EVALUATION_COMMAND, check=True, cwd=REPO)
    else:
        print('PENDING: command is printed only; set RUN_H1N_EXTERNAL_EVALUATION=1 after the freeze.')
else:
    print('PENDING: set H1N_FROZEN_EXPERIMENT and H1N_EXTERNAL_MANIFEST only after the documented freeze and licensed preparation.')

## Robustness status

The evaluator now restores the selected run's preprocessing through `ModelBundle` and applies every fixed JPEG/resize/blur condition **after** the H1-N common raster. The frozen checkpoint and validation-selected threshold must be reused; the clean internal predictions must never be overwritten. FPR@TPR=95% in a robustness table is a descriptive ROC-curve value computed from that condition's test scores, not a replacement operating threshold. The gated command below is H1-N-valid only after one experiment has been explicitly frozen. Its Defactify result remains an exploratory internal stress test because D0 already opened that test split.

In [ ]:
import os
import shlex
import subprocess
import sys

def robustness_command(frozen_experiment: str) -> list[str]:
    """Build the H1-N robustness command for one explicitly frozen experiment."""
    if Path(frozen_experiment).name != frozen_experiment:
        raise ValueError('Use one experiment directory name, not a path.')
    experiment_dir = REPO / 'artifacts' / frozen_experiment
    return [
        sys.executable,
        'scripts/evaluate_robustness.py',
        '--manifest', 'data/processed/defactify_grouped/manifest.csv',
        '--experiment-dir', str(experiment_dir.relative_to(REPO)),
        '--output-dir', str((experiment_dir / 'robustness').relative_to(REPO)),
    ]

FROZEN_EXPERIMENT = os.environ.get('H1N_FROZEN_EXPERIMENT')
if FROZEN_EXPERIMENT:
    ROBUSTNESS_COMMAND = robustness_command(FROZEN_EXPERIMENT)
    print(shlex.join(ROBUSTNESS_COMMAND))
    if os.environ.get('RUN_H1N_ROBUSTNESS') == '1':
        subprocess.run(ROBUSTNESS_COMMAND, check=True, cwd=REPO)
    else:
        print('PENDING: command is printed only; set RUN_H1N_ROBUSTNESS=1 after the freeze to run it.')
else:
    print('PENDING: set H1N_FROZEN_EXPERIMENT only after documenting the frozen checkpoint and threshold.')

In [ ]:
robustness_tables = []
for run in internal_runs:
    metrics_path = ARTIFACT_ROOT / str(run['run']) / 'robustness' / 'robustness_metrics.csv'
    if metrics_path.is_file():
        table = pd.read_csv(metrics_path)
        table.insert(0, 'run', run['run'])
        robustness_tables.append(table)

if robustness_tables:
    display(pd.concat(robustness_tables, ignore_index=True))
else:
    print('PENDING: no H1-N-verified robustness artifact is available.')

## Permitted conclusion

A high clean score with poor external or JPEG performance is evidence of distribution-specific artefacts, not a deployment-ready detector. A model score remains a score, not a calibrated probability or proof of origin. The web interface, if ever enabled, must expose this limitation and use only a model frozen after the external evaluation.

## 2. Results, aggregation, and limitations


This notebook reads saved artifacts rather than transcribing values. It deliberately separates D0 legacy diagnostics from H1-N controlled results. Empty H1-N tables mean that no relevant controlled experiment has been completed; they do not mean a metric is zero.

In [ ]:
import json

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL

artifact_root = REPO / 'artifacts'


def completed_neural_results(root: Path) -> pd.DataFrame:
    """Read only explicitly named H1-N neural run artifacts; never rediscover arbitrary runs."""
    rows = []
    for run_path in sorted(root.glob('h1n_*_resnet50_seed*/run.json')):
        metrics_path = run_path.parent / 'internal_test_metrics.json'
        if not metrics_path.is_file():
            continue
        run = json.loads(run_path.read_text(encoding='utf-8'))
        if run.get('preprocessing', {}).get('protocol') != CONTROLLED_PREPROCESSING_PROTOCOL:
            continue
        config = run.get('config', {})
        rows.append({
            'status': 'exploratory_internal_stress_test',
            'run': run_path.parent.name,
            'representation': config.get('representation'),
            'seed': config.get('seed'),
            'preprocessing': run['preprocessing']['protocol'],
            'image_size': run['preprocessing']['image_size'],
            'threshold': run.get('threshold'),
            **json.loads(metrics_path.read_text(encoding='utf-8')),
        })
    return pd.DataFrame(rows)


neural_results = completed_neural_results(artifact_root)
if neural_results.empty:
    print('PENDING: no completed H1-N neural internal result is available.')
else:
    display(neural_results.sort_values(['representation', 'seed']))

## D0 diagnostic ledger — not a result table for H1-N

These artifacts document the geometry/source confound and are retained for reproducibility. They are excluded from all controlled comparisons, seed aggregates, external-validation claims, model selection, and interface decisions.

In [ ]:
d0_rows = []
for run_name in ('radial_logistic_seed7', 'file_metadata_control_seed7'):
    metrics_path = artifact_root / run_name / 'internal_test_metrics.json'
    if metrics_path.is_file():
        d0_rows.append({
            'status': 'D0 diagnostic only — not H1-N evidence',
            'run': run_name,
            **json.loads(metrics_path.read_text(encoding='utf-8')),
        })

d0_results = pd.DataFrame(d0_rows)
if d0_results.empty:
    print('No D0 metric artifact is present.')
else:
    display(d0_results)

controlled_radial_dir = artifact_root / 'h1n_controls' / 'radial_fft_logistic_h1n_square_crop_128_v1_seed7'
controlled_radial_metrics = controlled_radial_dir / 'internal_test_metrics.json'
if controlled_radial_metrics.is_file():
    radial_result = pd.DataFrame([{
        'status': 'exploratory H1-N fixed-feature baseline',
        'run': controlled_radial_dir.name,
        **json.loads(controlled_radial_metrics.read_text(encoding='utf-8')),
    }])
    display(radial_result)
else:
    print('PENDING: no corrected controlled-radial baseline artifact is available.')

## Three-seed exploratory aggregation

Only after all three predeclared seeds have completed for both representations may the internal stress-test rows be aggregated. The aggregation is descriptive, not confirmatory: D0 already opened the same Defactify test split. `fpr_at_tpr_95` is likewise a descriptive value from each test-set ROC curve, not the validation-selected operating threshold recorded in `run.json`. Preserve per-seed rows, per-generator tables and cluster-bootstrap comparison artifacts beside this summary.

In [ ]:
aggregate_rows = []
for summary_path in sorted(artifact_root.glob('h1n_*_resnet50_aggregate/aggregate_summary.json')):
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    aggregate_rows.append({
        'aggregate': summary_path.parent.name,
        'status': summary.get('status'),
        'selection_rule': summary.get('selection_rule'),
        'metrics': summary.get('metrics'),
    })

if aggregate_rows:
    display(pd.DataFrame(aggregate_rows))
else:
    print('PENDING: run scripts/aggregate_experiments.py only after all three analyses per representation complete.')

## Confirmatory external and robustness ledger

The external Synthbuster + RAISE evaluation remains locked until the H1-N plan is frozen. Its generator-specific, macro and worst-generator metrics must be kept separate from Defactify internal stress-test values and stratified by the explicit prepared-manifest relation: same-named generator, same-family version, or unseen family. Thus it is inaccurate to call all Synthbuster rows unseen. Robustness rows are valid only if the evaluator has been checked to apply the same H1-N centre-crop and 128 × 128 raster before each fixed degradation.

In [ ]:
external_rows = []
for prediction_path in sorted(artifact_root.glob('h1n_*_resnet50_seed*/external/external_predictions.csv')):
    external_rows.append({
        'run': prediction_path.parent.parent.name,
        'path': str(prediction_path),
    })

if external_rows:
    display(pd.DataFrame(external_rows))
else:
    print('PENDING: no frozen external-evaluation prediction artifact exists; this is expected before the lock is released.')

## Limitations and permitted wording

- Geometry/source confounding required a protocol amendment; D0 must not be promoted to detector performance.
- H1-N is a controlled Defactify representation comparison, not universal image authentication.
- The original internal test is exploratory because it was seen during D0; confirmatory evidence requires the locked external corpus.
- FFT magnitude discards colour and phase and may still exploit corpus- or generator-specific artefacts.
- Model scores are not probabilities until a separately documented calibration procedure has been completed.

Use measured wording only: *On the specified exploratory internal stress test, [model] achieved [metric] under the H1-N protocol.* Only after the frozen external run may the report add: *On the separate locked Synthbuster + RAISE evaluation, the frozen model achieved [metric].* Never claim that the score proves an arbitrary image's origin.